# Week 05 Lab Work

**Dataset:** Adult / Census Income (UCI Machine Learning Repository)

1. Load the data.
2. Find the missing data.
3. Fill the missing data using an automatic technique.
4. Find the noisy data.
5. Handle the noisy data using an automatic technique.
6. Integrate the data from multiple sources.

In [1]:
import os
import urllib.request

import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer

cols = ["age", "workclass", "fnlwgt", "education", "education-num",
        "marital-status", "occupation", "relationship", "race", "sex",
        "capital-gain", "capital-loss", "hours-per-week", "native-country", "income"]


def get_file(data_filename):
    for base in [".", ".."]:
        if not os.path.isdir(base):
            continue
        for root, dirs, files in os.walk(base):
            if data_filename in files:
                return os.path.join(root, data_filename)

    print(f"'{data_filename}' not found in the workspace. Downloading it dynamically...")
    url = f"https://archive.ics.uci.edu/ml/machine-learning-databases/adult/{data_filename}"
    try:
        urllib.request.urlretrieve(url, data_filename)
        print("Download complete!")
        return data_filename
    except Exception as e:
        raise FileNotFoundError(
            f"Could not download the file automatically. Please manually upload '{data_filename}' "
            "to the Colab file explorer panel on the left."
        ) from e


# 1. Load the data
target_path = get_file("adult.data")
print(f"Loading dataset from: {target_path}")
df = pd.read_csv(target_path, header=None, names=cols, skipinitialspace=True, na_values="?")
df.head()

Loading dataset from: ../Task_No_3 (week 4)/adult dataset/adult.data


,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


In [2]:
# 2. Find missing data
print("Missing data per column:\n", df.isnull().sum())

# 3. Fill missing data automatically (median for numbers, mode for text)
numeric = df.select_dtypes(include=np.number).columns
categorical = df.columns.difference(numeric)

num_imputer = SimpleImputer(strategy="median")
cat_imputer = SimpleImputer(strategy="most_frequent")

df_filled = df.copy()
df_filled[numeric] = num_imputer.fit_transform(df[numeric])
df_filled[categorical] = cat_imputer.fit_transform(df[categorical])
print("\nMissing cells after filling:", df_filled.isnull().sum().sum())

# 4. Find noisy data using the IQR method
outliers_summary = {}
for col in numeric:
    Q1, Q3 = df_filled[col].quantile([0.25, 0.75])
    IQR = Q3 - Q1
    if IQR == 0:
        continue

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = df_filled[(df_filled[col] < lower_bound) | (df_filled[col] > upper_bound)]
    if not outliers.empty:
        outliers_summary[col] = len(outliers)

print("\nCount of noisy data (outliers) per column:")
print(outliers_summary)

# 5. Handle noisy data automatically using clipping
df_clean = df_filled.copy()
for col in outliers_summary:
    Q1, Q3 = df_clean[col].quantile([0.25, 0.75])
    IQR = Q3 - Q1
    df_clean[col] = np.clip(df_clean[col], Q1 - 1.5 * IQR, Q3 + 1.5 * IQR)

print("\nRange after clipping:")
print(df_clean[list(outliers_summary)].agg(["min", "max"]))

Missing data per column:
 age                  0
workclass         1836
fnlwgt               0
education            0
education-num        0
marital-status       0
occupation        1843
relationship         0
race                 0
sex                  0
capital-gain         0
capital-loss         0
hours-per-week       0
native-country     583
income               0
dtype: int64

Missing cells after filling: 0

Count of noisy data (outliers) per column:
{'age': 143, 'fnlwgt': 992, 'education-num': 1198, 'hours-per-week': 9008}

Range after clipping:
      age    fnlwgt  education-num  hours-per-week
min  17.0   12285.0            4.5            32.5
max  78.0  415887.0           16.0            52.5


In [3]:
# 6. Integrate data from multiple sources
test_path = get_file("adult.test")
print(f"Loading second source from: {test_path}")

df_test = pd.read_csv(test_path, header=None, names=cols, skiprows=1,
                      skipinitialspace=True, na_values="?")
df_test["income"] = df_test["income"].str.rstrip(".")
df_test[numeric] = num_imputer.transform(df_test[numeric])
df_test[categorical] = cat_imputer.transform(df_test[categorical])

integrated_df = pd.concat([df_clean, df_test], ignore_index=True).drop_duplicates()

print("Integrated dataset shape:", integrated_df.shape)
print("Missing cells:", integrated_df.isnull().sum().sum())
integrated_df.head()

Loading second source from: ../Task_No_3 (week 4)/adult dataset/adult.test


Integrated dataset shape: (48770, 15)
Missing cells: 0


,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39.0,State-gov,77516.0,Bachelors,13.0,Never-married,Adm-clerical,Not-in-family,White,Male,2174.0,0.0,40.0,United-States,<=50K
1,50.0,Self-emp-not-inc,83311.0,Bachelors,13.0,Married-civ-spouse,Exec-managerial,Husband,White,Male,0.0,0.0,32.5,United-States,<=50K
2,38.0,Private,215646.0,HS-grad,9.0,Divorced,Handlers-cleaners,Not-in-family,White,Male,0.0,0.0,40.0,United-States,<=50K
3,53.0,Private,234721.0,11th,7.0,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0.0,0.0,40.0,United-States,<=50K
4,28.0,Private,338409.0,Bachelors,13.0,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0.0,0.0,40.0,Cuba,<=50K
